# Pruebas de integración — Keithley + SmartLight (Vigo)

Ejecuta las celdas **en orden**, una etapa a la vez.

| Etapa | Qué prueba |
|-------|------------|
| 1 | Conexión SSH + medir_loop.py (sin Keithley) |
| 2 | Keithley solo (sin SmartLight) |
| 3 | Integración reducida (3 voltajes, 2 retos) |
| 4 | Barrido completo (solo si 1-3 funcionan) |


## Imports comunes
Ejecutar siempre primero.

In [11]:
import Meas_classes as ms
import paramiko
import json, time, getpass
import numpy as np
import pandas as pd

## Etapa 1 — Probar solo la conexión SSH + medir_loop.py
Comprueba que el SmartLight responde correctamente a un comando de prueba, sin tocar el Keithley.

In [12]:
PASSWORD = '29c6b8177fdf9b4523e93474d04a0ffa'

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect('10.42.0.125', username='smartlight', password=PASSWORD)

stdin, stdout, stderr = ssh.exec_command(
    'cd /home/smartlight/Oxel/VIGO && /home/smartlight/.venvs/ipronics/bin/python medir_loop.py'
)

def medir_remoto(inport, n_retos):
    cmd = {'inport': inport, 'n_retos': n_retos}
    stdin.write(json.dumps(cmd) + '\n')
    stdin.flush()
    return json.loads(stdout.readline())

print('✓ SSH conectado y medir_loop.py lanzado. Esperando a que el chip calibre...')


✓ SSH conectado y medir_loop.py lanzado. Esperando a que el chip calibre...


In [3]:
# Prueba con muy pocos retos para validar rápido
res_test = medir_remoto(inport=1, n_retos=2)
print(json.dumps(res_test, indent=2)[:1000])

{
  "inport": 1,
  "outports": [
    0,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    34,
    35,
    36,
    37,
    38,
    39
  ],
  "n_medidos": 2,
  "powers": [
    {
      "0": -39.990170004932715,
      "2": -40.42829126098343,
      "3": -38.36289698944425,
      "4": -39.61228420698523,
      "5": -41.526673918996636,
      "6": -39.839282993450944,
      "7": -39.47377281490877,
      "8": -38.79041640949335,
      "9": -40.10124852528399,
      "10": -40.52644133179136,
      "11": -38.99482775716955,
      "12": -39.797118199341874,
      "13": -41.08161456028696,
      "14": -39.81814958114216,
      "15": -38.80708835304945,
      "16": -40.356107600323625,
      "17": -40.80838320178779,
      "18": -39.2646632586562,
      "19": -38.454330451993755,
      "20": -40.75574124971648,
      "21": -40.83494580385967,
      "34": -38.27334844468766,
      "35": -39.53

In [6]:
try:
    ssh.close()
    print('✓ SSH cerrado')
except Exception as e:
    print(f'Ya estaba cerrado o hubo un error al cerrar: {e}')

✓ SSH cerrado


In [ ]:
print(stderr.read().decode())

Si ves un JSON con `'powers'` y dos diccionarios de potencias → **Etapa 1 OK**.

Si sale `'error'` o se queda colgado, revisa:
- ¿`retos_v5/challenges_ring_port21.json` existe en esa carpeta?
- ¿`logging.yaml` está en la misma carpeta?
- Mira `stderr.readlines()` en la celda siguiente para ver el motivo.

In [ ]:
# Solo si algo falla en la Etapa 1 — ver mensajes de error del SmartLight
# (ATENCIÓN: esto puede bloquearse si el proceso sigue vivo y no ha escrito nada nuevo)
# import select
# while ssh.get_transport().is_active():
#     line = stderr.readline()
#     if not line:
#         break
#     print(line.strip())

## Etapa 2 — Probar solo el Keithley
Sin tocar el SmartLight. Verifica que cambia de voltaje y lee bien.

In [4]:
keithley = ms.Keithley('20', 'GPIB0', 0.0011, -3)
keithley.set_keithley_for_meas_current()
keithley.on_keithley_fixed_volt()

('GPIB0::0::INSTR', 'GPIB0::20::INSTR')
KEITHLEY INSTRUMENTS INC.,MODEL 2401,4601558,B02 Jan 20 2021 10:19:49/B01  /W/N



In [5]:
VOLTAJES_TEST = [1, 3, 5]

for v in VOLTAJES_TEST:
    keithley.volt_source = v
    keithley.on_keithley_fixed_volt()
    time.sleep(0.3)
    V_medido, I_medido = keithley.read_keithley()
    print(f'V={v:.2f} V  →  V_medido={V_medido:.4f} V,  I={I_medido:.4f} mA')

keithley.off_keithley_volt()
print('\n✓ Etapa 2 completada')

V=1.00 V  →  V_medido=1.0000 V,  I=0.0000 mA
V=3.00 V  →  V_medido=3.0000 V,  I=0.0000 mA
V=5.00 V  →  V_medido=5.0000 V,  I=0.0000 mA

✓ Etapa 2 completada


Si los voltajes medidos son razonables (cercanos a los programados) → **Etapa 2 OK**.

## Etapa 3 — Integración reducida
3 voltajes × 2 retos. Junta Keithley + SmartLight ya probados por separado.

In [7]:
# Reconectar Keithley si la Etapa 2 ya lo apagó
keithley.on_keithley_fixed_volt()

VOLTAJES_REDUCIDO = [1, 3, 5]
INPORT  = 1
N_RETOS = 2

resultados_test = []

for v in VOLTAJES_REDUCIDO:
    keithley.volt_source = v
    keithley.on_keithley_fixed_volt()
    time.sleep(0.3)

    V_medido, I_medido = keithley.read_keithley()
    res_chip = medir_remoto(inport=INPORT, n_retos=N_RETOS)

    if 'error' in res_chip:
        print(f'⚠ V={v:.2f} V → error SmartLight: {res_chip["error"]}')
        continue

    resultados_test.append({
        'voltaje_programado': v,
        'voltaje_medido':     V_medido,
        'corriente_mA':       I_medido,
        'powers':             res_chip['powers'],
    })

    print(f'V={v:.2f} V  →  V_medido={V_medido:.4f} V,  '
          f'I={I_medido:.4f} mA,  {res_chip["n_medidos"]} retos medidos')

print(f'\n✓ Etapa 3 completada — {len(resultados_test)} puntos')

V=1.00 V  →  V_medido=1.0000 V,  I=-0.0000 mA,  2 retos medidos
V=3.00 V  →  V_medido=3.0000 V,  I=0.0000 mA,  2 retos medidos
V=5.00 V  →  V_medido=5.0000 V,  I=0.0000 mA,  2 retos medidos

✓ Etapa 3 completada — 3 puntos


Si esto funciona sin errores → **Etapa 3 OK**, ya puedes lanzar el barrido completo.

## Etapa 4 — Barrido completo
Ejecutar solo cuando las etapas 1-3 hayan funcionado correctamente.

In [13]:
VOLTAJES    = np.linspace(1, 5, 9)   # ajusta tu rango real
INPORT      = 1
N_RETOS     = 5

resultados = []

keithley.on_keithley_fixed_volt()

for v in VOLTAJES:
    keithley.volt_source = v
    keithley.on_keithley_fixed_volt()
    time.sleep(0.3)

    V_medido, I_medido = keithley.read_keithley()
    res_chip = medir_remoto(inport=INPORT, n_retos=N_RETOS)

    if 'error' in res_chip:
        print(f'⚠ V={v:.2f} V → error SmartLight: {res_chip["error"]}')
        continue

    resultados.append({
        'voltaje_programado': v,
        'voltaje_medido':     V_medido,
        'corriente_mA':       I_medido,
        'powers':             res_chip['powers'],
    })

    print(f'V={v:.2f} V  →  V_medido={V_medido:.4f} V,  '
          f'I={I_medido:.4f} mA,  {res_chip["n_medidos"]} retos medidos')

V=1.00 V  →  V_medido=1.0000 V,  I=0.0000 mA,  5 retos medidos
V=1.50 V  →  V_medido=1.5000 V,  I=0.0000 mA,  5 retos medidos
V=2.00 V  →  V_medido=2.0000 V,  I=0.0000 mA,  5 retos medidos
V=2.50 V  →  V_medido=2.5000 V,  I=-0.0000 mA,  5 retos medidos
V=3.00 V  →  V_medido=3.0000 V,  I=0.0000 mA,  5 retos medidos
V=3.50 V  →  V_medido=3.5000 V,  I=-0.0000 mA,  5 retos medidos
V=4.00 V  →  V_medido=4.0000 V,  I=0.0000 mA,  5 retos medidos
V=4.50 V  →  V_medido=4.5000 V,  I=-0.0000 mA,  5 retos medidos
V=5.00 V  →  V_medido=5.0000 V,  I=0.0000 mA,  5 retos medidos


## Cierre limpio
Ejecutar **siempre al terminar**, aunque haya habido errores.

In [14]:
keithley.off_keithley_volt()

stdin.write('EXIT\n')
stdin.flush()
ssh.close()

print('✓ Keithley apagado, SmartLight desconectado, SSH cerrado.')

✓ Keithley apagado, SmartLight desconectado, SSH cerrado.


## Guardar resultados
Ejecutar tras la Etapa 4 (barrido completo).

In [15]:
with open('barrido_voltaje_chip_v2.json', 'w') as f:
    json.dump(resultados, f, indent=2)

df_resumen = pd.DataFrame([
    {'voltaje_programado': r['voltaje_programado'],
     'voltaje_medido':     r['voltaje_medido'],
     'corriente_mA':       r['corriente_mA'],
     'n_retos':            len(r['powers'])}
    for r in resultados
])
df_resumen.to_csv('barrido_voltaje_resumen.csv', index=False)

print(f'✓ {len(resultados)} puntos guardados en barrido_voltaje_chip.json')
print(df_resumen)

✓ 9 puntos guardados en barrido_voltaje_chip.json
   voltaje_programado  voltaje_medido  corriente_mA  n_retos
0                 1.0             1.0  1.301355e-10        5
1                 1.5             1.5  4.583486e-11        5
2                 2.0             2.0  8.467445e-10        5
3                 2.5             2.5 -4.600209e-10        5
4                 3.0             3.0  5.938236e-10        5
5                 3.5             3.5 -7.972477e-10        5
6                 4.0             4.0  1.099669e-09        5
7                 4.5             4.5 -5.443406e-10        5
8                 5.0             5.0  6.781312e-10        5
